# E1 — Cuantización: Trade-off velocidad vs precisión

## Pregunta de investigación

¿Cómo afecta la cuantización al trade-off entre velocidad de inferencia y calidad de salida?

## Hipótesis previa

Esperamos: mayor compresión (menor BPW) → mayor throughput pero mayor perplejidad.

## Configuración del experimento

| Parámetro | Valores |
|-----------|---------|
| Modelo | Llama-3.2-3B |
| Cuantizaciones | 4 (Q2, Q3, Q4, F32) |
| Total | 4 runs |


In [1]:
"""Setup: carga y filtrado de runs para E1."""

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

from monitorviz.io import load_collection
from monitorviz.viz import setup_style

setup_style("talk")
warnings.filterwarnings("ignore", category=UserWarning, module="seaborn")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# --- Carga de datos ---
_here = Path.cwd()
PROJECT_ROOT = _here.parent if _here.name == "notebooks" else _here
DATA_ROOT = PROJECT_ROOT / "data" / "tfg-data"

assert DATA_ROOT.is_dir(), f"❌ No encontrado: {DATA_ROOT.resolve()}"

coll = load_collection(DATA_ROOT)
print(f"Runs cargados totales: {len(coll)}")

# --- Filtrado para E1 ---
# TODO: Definir el filtrado específico para este experimento
summary = coll.summary_df()
print(f"\nRuns en {summary.shape[0]}")
print(summary[["model_label", "engine", "fan", "accelerator"]].drop_duplicates())


Runs cargados totales: 120

Runs en 120
             model_label  engine    fan  accelerator
0          ministral (O)  OLLAMA  False        False
1          ministral (L)   LLAMA  False        False
2       Llama-3.2-3B (O)  OLLAMA  False        False
3       Llama-3.2-3B (L)   LLAMA  False        False
4       Llama-3.2-1B (O)  OLLAMA  False        False
5       Llama-3.2-1B (L)   LLAMA  False        False
6            granite (O)  OLLAMA  False        False
7            granite (L)   LLAMA  False        False
8              gemma (O)  OLLAMA  False        False
9              gemma (L)   LLAMA  False        False
10        deepseek:q (O)  OLLAMA  False        False
11  DeepSeek-R1-1.5B (L)   LLAMA  False        False
12         ministral (O)  OLLAMA   True        False
13         ministral (L)   LLAMA   True        False
14      Llama-3.2-3B (O)  OLLAMA   True        False
15      Llama-3.2-3B (L)   LLAMA   True        False
16      Llama-3.2-1B (O)  OLLAMA   True        False
17    

## Resumen ejecutivo

> TODO: Actualizar con resultados finales.

Placeholder: 3-5 frases resumiendo los hallazgos principales de E1.


## Comparativa de métricas de inferencia

Throughput, latencia, TTFT, perplejidad por configuración.


In [2]:
# Tabla resumen
display_cols = [
    "model_label", "engine",
    "tokens_per_s_mean", "words_per_s_mean",
    "latency_ms_mean", "ttft_ms_mean",
    "perplexity_geomean",
]
cols = [c for c in display_cols if c in summary.columns]
summary[cols].round(3)


,model_label,engine,tokens_per_s_mean,words_per_s_mean,latency_ms_mean,ttft_ms_mean,perplexity_geomean
0,ministral (O),OLLAMA,2.100,1.473,424964.275,20014.089,2.103
1,ministral (L),LLAMA,2.858,2.010,407140.417,3629.400,2.810
2,Llama-3.2-3B (O),OLLAMA,2.554,1.861,163999.251,12763.883,1.430
3,Llama-3.2-3B (L),LLAMA,2.384,1.740,850692.187,4291.243,2.286
4,Llama-3.2-1B (O),OLLAMA,6.830,5.137,60048.807,4603.280,1.654
...,...,...,...,...,...,...,...
115,gemma (L),LLAMA,5.696,3.962,99428.910,2608.276,2.420
116,granite (L),LLAMA,4.403,3.396,88554.701,2348.991,2.854
117,Llama-3.2-1B (L),LLAMA,11.246,8.675,209229.149,674.234,2.360
118,Llama-3.2-3B (L),LLAMA,4.614,3.379,465015.352,1933.339,2.286


## Comparativa de hardware

Temperatura, frecuencia, CPU%, potencia, throttling.


In [3]:
if "temp_max_c" in summary.columns:
    hw_cols = [
        "model_label",
        "temp_max_c", "temp_mean_c",
        "power_mean_w", "power_max_w",
        "throttled_ratio",
    ]
    hw_cols = [c for c in hw_cols if c in summary.columns]
    summary[hw_cols].round(2)
else:
    print("Sin datos de hardware")


## Análisis específico — E1

> TODO: Sección específica del experimento (sensibilidad, comparativa, etc.).


## Trade-offs multidimensionales — Frontera de Pareto

> TODO: Visualizar pareto_panel_multi si procede.


## Eficiencia computacional y energética

W_CPU, η_CPU, desglose por fases, core·s/J, energía/token.


In [4]:
if "cpu_work_core_s" in summary.columns:
    eff_cols = [
        "model_label",
        "cpu_work_core_s",
        "cpu_efficiency",
        "energy_per_token_j",
    ]
    eff_cols = [c for c in eff_cols if c in summary.columns]
    summary[eff_cols].round(3)
else:
    print("Sin datos de eficiencia CPU")


## Rendimiento vs parámetro experimental

Scatter del parámetro principal del experimento vs métricas clave.

> TODO: Implementar según el experimento concreto.


## Diagramas polares comparativos

Radar de rendimiento y hardware.

> TODO: Implementar radar_chart similar a 04_comparativa_global.


## Matriz de correlación de Pearson

> TODO: Implementar si n ≥ 3 runs.


## Alertas y anomalías detectadas


In [5]:
alerts = []

throttled = summary[summary["throttled_ratio"] > 0.5]
if not throttled.empty:
    for _, row in throttled.iterrows():
        alerts.append(f"⚠️  {{row['model_label']}}: {{row['throttled_ratio']*100:.1f}}% throttling")

if alerts:
    for a in alerts:
        print(a)
else:
    print("✅ Sin alertas")


⚠️  {row['model_label']}: {row['throttled_ratio']*100:.1f}% throttling
⚠️  {row['model_label']}: {row['throttled_ratio']*100:.1f}% throttling
⚠️  {row['model_label']}: {row['throttled_ratio']*100:.1f}% throttling
⚠️  {row['model_label']}: {row['throttled_ratio']*100:.1f}% throttling
⚠️  {row['model_label']}: {row['throttled_ratio']*100:.1f}% throttling
⚠️  {row['model_label']}: {row['throttled_ratio']*100:.1f}% throttling
⚠️  {row['model_label']}: {row['throttled_ratio']*100:.1f}% throttling
⚠️  {row['model_label']}: {row['throttled_ratio']*100:.1f}% throttling
⚠️  {row['model_label']}: {row['throttled_ratio']*100:.1f}% throttling
⚠️  {row['model_label']}: {row['throttled_ratio']*100:.1f}% throttling
⚠️  {row['model_label']}: {row['throttled_ratio']*100:.1f}% throttling
⚠️  {row['model_label']}: {row['throttled_ratio']*100:.1f}% throttling
⚠️  {row['model_label']}: {row['throttled_ratio']*100:.1f}% throttling
⚠️  {row['model_label']}: {row['throttled_ratio']*100:.1f}% throttling
⚠️  {r

## Conclusiones y discusión

### Confirmación/refutación de hipótesis

> TODO: Actualizar con resultados.

### Hallazgos principales

> TODO: Lista de descubrimientos.

### Limitaciones

> TODO: Limitaciones metodológicas.

### Implicaciones para el TFG

> TODO: Cómo contribuye este experimento a las conclusiones finales.

### Cuestiones abiertas

> TODO: Preguntas para trabajos futuros.
